# NRGA-Net — Full-dataset table generator, Drive-direct local (Colab Pro+)

This notebook trains NRGA-Net on the **complete training split** stored in your Google Drive and evaluates on the independent final-test split. **No data is copied to Colab local SSD and no dataset is uploaded to GitHub.** Everything reads directly from `MyDrive/PhD_GIS_Security/DB_local`.

**Recommended runtime:** GPU (A100 or V100), High-RAM enabled.

**What it does:**
1. Clones the NRGA-Net repository (code only).
2. Mounts your Google Drive.
3. Generates deterministic `train / calibration / final-test` splits from the Drive dataset.
4. Runs the full training run reading directly from Drive.
5. Saves checkpoints to `DB_local/NRGA_Net_revised` every few epochs.
6. Evaluates on the final-test set and produces Tables 3–9, saved back to Drive.

**Why Drive-direct?**
For very large datasets, copying to `/content` can take hours and is lost on reconnect. Reading directly from Drive is slower per epoch but avoids the copy and survives reconnects.

**After running:**
- All CSV tables are in `MyDrive/PhD_GIS_Security/DB_local/NRGA_Net_revised/full_tables/`.
- The best checkpoint is `MyDrive/PhD_GIS_Security/DB_local/NRGA_Net_revised/nrga_full_best.pt`.
- Replace the `[TO BE FILLED]` placeholders in the manuscript with these numbers.


In [ ]:
# ------------------------------------------------------------------------------
# Cell 1: clone repository and install dependencies (code only, no data)
# ------------------------------------------------------------------------------
!git clone --depth 1 https://github.com/haidarraad-a11y/NRGA-Net.git NRGA-Net
%cd NRGA-Net
!pip install -q -r requirements.txt


In [ ]:
# ------------------------------------------------------------------------------
# Cell 2: mount Google Drive and configure Drive-direct paths
# ------------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ============================================================================
# EDIT ONLY THIS BLOCK if your Drive layout differs from the existing notebook.
# These paths match NRGA_Net_Training_DenseNet201_v12_noViT_VANKv15.ipynb.
# ============================================================================

# Base dataset directory (same as BASE_DB in your existing notebook)
DATA_ROOT = '/content/drive/MyDrive/PhD_GIS_Security/DB_local'

# Your existing pipeline's output folder (kept unchanged)
EXISTING_OUTPUT_DIR = f'{DATA_ROOT}/NRGA_Local_V15'

# New folder for the revised-run checkpoints and tables, kept under DB_local
RUN_OUTPUT_DIR = f'{DATA_ROOT}/NRGA_Net_revised'

# Repository path inside Colab (code only, no data)
REPO_ROOT = '/content/NRGA-Net'
# ============================================================================

import os
os.environ['NRGA_DATA_ROOT'] = DATA_ROOT
os.environ['NRGA_TEST_ROOT'] = DATA_ROOT
os.environ['NRGA_QUICK_MODE'] = '0'   # full run, no shortcuts

print('DATA_ROOT:', DATA_ROOT)
print('EXISTING_OUTPUT_DIR:', EXISTING_OUTPUT_DIR)
print('RUN_OUTPUT_DIR:', RUN_OUTPUT_DIR)
print('REPO_ROOT:', REPO_ROOT)
print('NRGA_DATA_ROOT:', os.environ['NRGA_DATA_ROOT'])


In [ ]:
# ------------------------------------------------------------------------------
# Cell 3: validate dataset layout and generate deterministic splits
# ------------------------------------------------------------------------------
import subprocess, sys, json
from pathlib import Path

expected = [
    'Fake-Vaihingen/real/train',
    'Fake-Vaihingen/fake/train/lama',
    'Fake-Vaihingen/fake/train/repaint',
    'Fake-LoveDA/real/train',
    'Fake-LoveDA/fake/train/lama',
    'Fake-LoveDA/fake/train/repaint',
    'Local_Diffusion/real/train',
    'Local_Diffusion/fake/train',
]

missing = []
for rel in expected:
    p = Path(DATA_ROOT) / rel
    if not p.exists():
        missing.append(str(p))

if missing:
    print('WARNING: the following expected folders are missing:')
    for m in missing:
        print('  ', m)
    print('Please check DATA_ROOT in Cell 2.')
else:
    print('Dataset layout looks correct.')

subprocess.run([sys.executable, 'scripts/create_splits.py',
                '--root', DATA_ROOT,
                '--seed', '42',
                '--cal-frac', '0.20',
                '--out', f'{REPO_ROOT}/splits'], check=True)

with open(f'{REPO_ROOT}/splits/metadata.json') as f:
    meta = json.load(f)
print(json.dumps(meta['splits'], indent=2))


In [ ]:
# ------------------------------------------------------------------------------
# Cell 4: verify Drive-direct access (no local copy)
# ------------------------------------------------------------------------------
from pathlib import Path

DATA_ROOT = '/content/drive/MyDrive/PhD_GIS_Security/DB_local'

# Count images directly in Drive to confirm access
img_exts = {'.png','.jpg','.jpeg','.bmp','.tif','.tiff'}
n_train_vaihingen = sum(1 for p in (Path(DATA_ROOT)/'Fake-Vaihingen').rglob('*') if p.suffix.lower() in img_exts)
n_train_loveda    = sum(1 for p in (Path(DATA_ROOT)/'Fake-LoveDA').rglob('*') if p.suffix.lower() in img_exts)
n_train_localdiff = sum(1 for p in (Path(DATA_ROOT)/'Local_Diffusion').rglob('*') if p.suffix.lower() in img_exts)

print(f'Images found in Drive:')
print(f'  Fake-Vaihingen: {n_train_vaihingen}')
print(f'  Fake-LoveDA:    {n_train_loveda}')
print(f'  Local_Diffusion:{n_train_localdiff}')
print('\nTraining will read directly from these Drive folders (no local SSD copy).')


In [ ]:
# ------------------------------------------------------------------------------
# Cell 5: import all definitions from src/main.py without running the full pipeline
# ------------------------------------------------------------------------------
import os, sys
REPO_ROOT = '/content/NRGA-Net'
RUN_OUTPUT_DIR = '/content/drive/MyDrive/PhD_GIS_Security/DB_local/NRGA_Net_revised'
sys.path.insert(0, f'{REPO_ROOT}/src')

main_path = f'{REPO_ROOT}/src/main.py'
with open(main_path, 'r', encoding='utf-8') as f:
    code_all = f.read()

prefix = code_all.split('# --- NRGA-NOTEBOOK-DEFINITIONS-END ---')[0]
print(f'Executing {len(prefix.splitlines())} lines of definitions from src/main.py...')
exec(prefix)

# Override the output directory so the revised run writes under DB_local
# while keeping your existing NRGA_Local_V15 folder untouched.
cfg.OUTPUT_DIR = RUN_OUTPUT_DIR
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
print('OUTPUT_DIR overridden to:', cfg.OUTPUT_DIR)

# Reduced epoch budget: resume-capable runs stop at 100 instead of 200.
cfg.EPOCHS = 100
print('EPOCHS overridden to:', cfg.EPOCHS)

print('Definitions loaded.')
print('Train loader length:', len(train_loader) if 'train_loader' in globals() else 'N/A')
print('Val   loader length:', len(val_loader) if 'val_loader' in globals() else 'N/A')


In [ ]:
# ------------------------------------------------------------------------------
# Cell 5.5: create optimizer, scheduler, scaler, and EMA
# ------------------------------------------------------------------------------
import torch
from torch.cuda.amp import GradScaler

optimizer = optim.AdamW(build_param_groups(model, cfg),
                        lr=float(getattr(cfg, 'LR_DECODER', 5e-4)),
                        weight_decay=cfg.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS, eta_min=1e-6)
scaler = GradScaler()
ema = EMA(model, getattr(cfg, 'EMA_DECAY', 0.999)) if getattr(cfg, 'EMA_ENABLE', True) else None

print('optimizer:', type(optimizer).__name__)
print('scheduler:', type(scheduler).__name__)
print('scaler:', type(scaler).__name__)
print('ema:', ema)


In [ ]:
# ------------------------------------------------------------------------------
# Cell 6: training with checkpoint backups to Drive every few epochs
# ------------------------------------------------------------------------------
import torch
from pathlib import Path
from collections import defaultdict

RUN_OUTPUT_DIR = '/content/drive/MyDrive/PhD_GIS_Security/DB_local/NRGA_Net_revised'
Path(RUN_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

best_score = 0.0
patience_counter = 0
patience = 10
history = defaultdict(list)
output_dir = Path(cfg.OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
best_path = output_dir / 'nrga_full_best.pt'

# Optional: resume from previous revised-run checkpoint if it exists
if best_path.exists():
    print('Resuming from:', best_path)
    ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    start_epoch = ckpt.get('epoch', 0) + 1
else:
    start_epoch = 1

for epoch in range(start_epoch, cfg.EPOCHS + 1):
    model.train()
    train_loss, _ = train_one_epoch(model, train_loader, optimizer, scaler, criterion, ema=ema)
    if ema is not None:
        ema.apply_to(model)
    val_m, *_ = validate(model, val_loader, criterion, tta_ms=False)
    if ema is not None:
        ema.restore(model)

    score = val_m.get('pooled_iou', val_m.get('mean_iou', 0.0))
    history['train_loss'].append(train_loss.get('total', 0.0))
    history['val_pooled_iou'].append(score)
    print(f'Epoch {epoch:02d}/{cfg.EPOCHS}  train_loss={train_loss.get("total",0):.4f}  '
          f'val_pooled_iou={score:.4f}  val_det_acc={val_m.get("accuracy",0):.4f}')

    if score > best_score:
        best_score = score
        patience_counter = 0
        torch.save({'model_state': model.state_dict(), 'cfg': cfg, 'epoch': epoch}, best_path)
        print('  -> new best saved')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch}')
            break

    # Periodic backup every 10 epochs (in case of disconnect)
    if epoch % 10 == 0:
        backup_path = Path(RUN_OUTPUT_DIR) / f'nrga_full_epoch{epoch:03d}.pt'
        torch.save({'model_state': model.state_dict(), 'cfg': cfg, 'epoch': epoch}, backup_path)
        print(f'  -> periodic backup saved: {backup_path}')

# Load best checkpoint
ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print('Best checkpoint loaded from', best_path)


In [ ]:
# ------------------------------------------------------------------------------
# Cell 7: optional 384 px fine-tune -- starts from the saved 256 best checkpoint,
# so the 256 training does NOT need to be re-run. Resumable across disconnects.
# ------------------------------------------------------------------------------
import copy as _copy, time
from pathlib import Path
from torch.utils.data import DataLoader, ConcatDataset
from torch.cuda.amp import GradScaler

if not getattr(cfg, 'FT384_ENABLE', True):
    print('FT384_ENABLE=False -> skipping 384 fine-tune; evaluation will use the 256 model.')
else:
    FT_SIZE   = int(getattr(cfg, 'FT384_IMG_SIZE', 384))
    FT_EPOCHS = int(getattr(cfg, 'FT384_EPOCHS', 40))
    FT_LR     = float(getattr(cfg, 'FT384_LR', 2e-5))
    FT_BATCH  = int(getattr(cfg, 'FT384_BATCH', 3))
    FT_ACCUM  = int(getattr(cfg, 'FT384_ACCUM', 2))
    FT_PAT    = int(getattr(cfg, 'FT384_PATIENCE', 4))

    best256_path = Path(RUN_OUTPUT_DIR) / 'nrga_full_best.pt'
    best384_path = Path(RUN_OUTPUT_DIR) / 'nrga_full_384_best.pt'
    last384_path = Path(RUN_OUTPUT_DIR) / 'nrga_full_384_last.pt'
    assert best256_path.exists(), '256 best checkpoint not found - run Cell 6 first'

    # 384 datasets built from the same Drive folders as the 256 stage
    _tr384, _va384 = [], []
    _seen384 = {'train': set(), 'val': set()}
    for ds_name, splits in cfg.DATASET_PATHS.items():
        for split_name, split_key, ds_list, do_aug in [('train', 'train', _tr384, True),
                                                       ('val', 'val', _va384, False)]:
            if split_key in splits:
                paths = splits[split_key]
                _rk = os.path.realpath(str(paths['real']))
                _lr = (not getattr(cfg, 'DEDUPE_REALS', True)) or (_rk not in _seen384[split_name])
                _seen384[split_name].add(_rk)
                ds_list.append(InpaintingSegDataset(
                    real_dir=paths['real'], fake_dir=paths['fake'], mask_dir=paths['mask'],
                    dataset_name=ds_name, split=split_name, img_size=FT_SIZE,
                    augment=do_aug, native_crop=getattr(cfg, 'NATIVE_CROP', False),
                    load_reals=_lr))
    train_loader384 = DataLoader(ConcatDataset(_tr384), batch_size=FT_BATCH, shuffle=True,
                                 num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader384 = DataLoader(ConcatDataset(_va384), batch_size=FT_BATCH, shuffle=False,
                               num_workers=cfg.NUM_WORKERS, pin_memory=True)
    print(f'384 loaders: {len(train_loader384)} train batches, {len(val_loader384)} val batches')

    cfg384 = _copy.copy(cfg)
    cfg384.IMG_SIZE = FT_SIZE
    cfg384.BATCH_SIZE = FT_BATCH
    model384 = NRGANet(cfg384).to(DEVICE)
    _ck = torch.load(best256_path, map_location=DEVICE, weights_only=False)
    model384.load_state_dict(_ck['model_state'])
    if sanitize_state_(model384) > 0:
        print(' WARNING: the 256 base checkpoint contained non-finite tensors (neutralised).')
    print(f'Loaded 256 best (epoch {_ck.get("epoch", "?")}) -> fine-tuning at {FT_SIZE}px')
    for _name in ('content_prior', 'masked_prior'):
        _m = getattr(model384, _name, None)
        if _m is not None:
            _m.eval()
            for _p in _m.parameters():
                _p.requires_grad = False
    if hasattr(model384.cbfh, 'set_temperature'):
        model384.cbfh.set_temperature()

    _cfg_ft = _copy.copy(cfg)
    _cfg_ft.LR_ENCODER = FT_LR
    _cfg_ft.LR_DECODER = FT_LR * float(getattr(cfg, 'FT384_DECODER_MULT', 10.0))
    opt384 = optim.AdamW(build_param_groups(model384, _cfg_ft),
                         lr=_cfg_ft.LR_DECODER, weight_decay=cfg.WEIGHT_DECAY)
    sched384 = optim.lr_scheduler.CosineAnnealingLR(opt384, T_max=FT_EPOCHS, eta_min=1e-6)
    scaler384 = GradScaler()
    ema384 = EMA(model384, getattr(cfg, 'EMA_DECAY', 0.999)) if getattr(cfg, 'EMA_ENABLE', True) else None
    crit384 = NRGALoss(cfg384).to(DEVICE)

    best384_score, patience384, start384 = -1.0, 0, 1
    if getattr(cfg, 'RESUME', True) and last384_path.exists() and not getattr(cfg, 'FT384_RESET', False):
        _r4 = torch.load(last384_path, map_location=DEVICE, weights_only=False)
        _bad4 = [k for k, v in _r4['model_state'].items()
                 if torch.is_floating_point(v) and not bool(torch.isfinite(v).all())]
        if _bad4:
            print(f'>>> FT384 resume file has {len(_bad4)} non-finite tensors -> restarting from 256 base')
        else:
            model384.load_state_dict(_r4['model_state'])
            opt384.load_state_dict(_r4['optim_state'])
            sched384.load_state_dict(_r4['sched_state'])
            scaler384.load_state_dict(_r4['scaler_state'])
            if ema384 is not None and _r4.get('ema_shadow') is not None:
                ema384.shadow = {k: v.to(DEVICE) for k, v in _r4['ema_shadow'].items()}
                ema384.updates = int(_r4.get('ema_updates', 0))
                sanitize_ema_(ema384)
            start384 = int(_r4['epoch']) + 1
            best384_score = float(_r4.get('best_score', -1.0))
            patience384 = int(_r4.get('patience', 0))
            print(f'>>> FT384 resumed at epoch {start384} (best score {best384_score:.4f})')

    print(f'384 fine-tune: {FT_EPOCHS} epochs | batch {FT_BATCH}x{FT_ACCUM} accum | lr {FT_LR} | patience {FT_PAT}')
    for epoch in range(start384, FT_EPOCHS + 1):
        _t0 = time.time()
        train_loss, _ = train_one_epoch(model384, train_loader384, opt384,
                                        scaler384, crit384, ema384, FT_ACCUM)
        ema_active = ema384 is not None and ema384.updates >= getattr(cfg, 'EMA_WARMUP_STEPS', 300)
        if ema_active:
            ema384.apply_to(model384)
        val_m, *_ = validate(model384, val_loader384, crit384, tta_ms=False)
        cur_score = val_m.get('pooled_iou', val_m.get('mean_iou', 0.0))
        if not np.isfinite(cur_score) or cur_score <= 0.0:
            print(f'FT384 {epoch}/{FT_EPOCHS} -> score {cur_score} (non-finite or zero) -- not saved')
            cur_score = -1.0
        print(f'FT384 {epoch}/{FT_EPOCHS} ({time.time()-_t0:.0f}s)  '
              f'train_loss={train_loss.get("total",0):.4f}  '
              f'val_pooled_iou={cur_score:.4f}  val_det_acc={val_m.get("accuracy",0):.4f}')
        if cur_score > best384_score:
            best384_score, patience384 = cur_score, 0
            torch.save({'epoch': epoch, 'model_state': model384.state_dict(),
                        'metrics': val_m, 'img_size': int(FT_SIZE)}, best384_path)
            print('  -> new best 384 saved')
        else:
            patience384 += 1
            print(f'  no improvement ({patience384}/{FT_PAT})')
        if ema_active:
            ema384.restore(model384)
        sched384.step()
        torch.save({'epoch': epoch, 'model_state': model384.state_dict(),
                    'img_size': int(FT_SIZE),
                    'optim_state': opt384.state_dict(),
                    'sched_state': sched384.state_dict(),
                    'scaler_state': scaler384.state_dict(),
                    'ema_shadow': (ema384.shadow if ema384 is not None else None),
                    'ema_updates': (ema384.updates if ema384 is not None else 0),
                    'best_score': best384_score, 'patience': patience384}, last384_path)
        if patience384 >= FT_PAT:
            print('Early stopping (384 fine-tune).')
            break

    # hand the fine-tuned model to the evaluation cells (Cells 8-11)
    if best384_path.exists():
        _b4 = torch.load(best384_path, map_location=DEVICE, weights_only=False)
        model384.load_state_dict(_b4['model_state'])
        model = model384
        cfg.IMG_SIZE = FT_SIZE  # Cell 8 builds the test loader at this size
        print(f'384 fine-tune done (best pooled IoU {best384_score:.4f}). '
              f'Evaluation cells will use the 384 model.')
    else:
        print('384 fine-tune produced no best checkpoint; evaluation stays on the 256 model.')


In [ ]:
# ------------------------------------------------------------------------------
# Cell 8: calibration / final-test loaders from the immutable split files,
#         plus protocol-compliant checkpoint selection on the calibration split ONLY
# ------------------------------------------------------------------------------
import numpy as np, os, shutil, json, copy as _copy
from pathlib import Path
from torch.utils.data import DataLoader, ConcatDataset
import pandas as pd

REPO_ROOT = '/content/NRGA-Net'
SPLITS_DIR = Path(REPO_ROOT) / 'splits'
LINK_BASE = Path('/content/eval_links')

def _link_all(dst_dir, pairs):
    dst_dir.mkdir(parents=True, exist_ok=True)
    for src, name in pairs:
        d = dst_dir / name
        if d.exists() or d.is_symlink():
            d.unlink()
        os.symlink(src, d)

def build_loader_from_split(csv_path, tag, img_size=None):
    df = pd.read_csv(csv_path)
    groups = {}
    for _, r in df.iterrows():
        bench = str(r['benchmarks']).split(';')[0]
        fam = str(r['families']).split(';')[0] if isinstance(r['families'], str) and r['families'] else 'real'
        key = (bench, fam)
        if key not in groups:
            groups[key] = {'real': [], 'fake': []}
        if int(r['label']) == 1:
            m = str(r['mask_path']) if isinstance(r['mask_path'], str) and r['mask_path'] else None
            groups[key]['fake'].append((str(r['path']), m))
        else:
            groups[key]['real'].append(str(r['path']))
    base = LINK_BASE / tag
    if base.exists():
        shutil.rmtree(base)
    dss, names = [], []
    for (bench, fam), items in sorted(groups.items()):
        gname = f'{bench}-{fam}' if fam != 'real' else bench
        rd, fd, md = base / gname / 'real', base / gname / 'fake', base / gname / 'mask'
        _link_all(rd, [(p, f'{i:05d}_{Path(p).name}') for i, p in enumerate(items['real'])])
        with_mask = [(p, m) for (p, m) in items['fake'] if m is not None]
        _link_all(fd, [(p, f'{Path(p).stem}{Path(p).suffix}') for p, _ in with_mask])
        _link_all(md, [(m, f'{Path(p).stem}_mask{Path(m).suffix}') for (p, m) in with_mask])
        ds = InpaintingSegDataset(
            real_dir=str(rd), fake_dir=str(fd), mask_dir=str(md),
            dataset_name=gname, split='val', img_size=img_size or cfg.IMG_SIZE,
            augment=False, native_crop=getattr(cfg, 'NATIVE_CROP', False), load_reals=True)
        if len(ds) > 0:
            dss.append(ds)
            names.append(gname)
    loader = DataLoader(ConcatDataset(dss), batch_size=cfg.BATCH_SIZE, shuffle=False,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True)
    return loader, names

def _split_counts(csv_path):
    df = pd.read_csv(csv_path)
    return {'total': int(len(df)),
            'real': int((df['label'] == 0).sum()),
            'fake': int((df['label'] == 1).sum())}

results_dir = Path(REPO_ROOT) / 'results' / 'full_tables'
results_dir.mkdir(parents=True, exist_ok=True)

test_loader, test_names = build_loader_from_split(SPLITS_DIR / 'test.csv', 'test')
cal_loader, cal_names = build_loader_from_split(SPLITS_DIR / 'calibration.csv', 'cal')
print('Final-test datasets:', test_names)
print('Calibration datasets:', cal_names)
with open(results_dir / 'splits_summary.json', 'w') as f:
    json.dump({'calibration': _split_counts(SPLITS_DIR / 'calibration.csv'),
               'test': _split_counts(SPLITS_DIR / 'test.csv')}, f, indent=2)

# ---- protocol-compliant checkpoint selection (calibration split ONLY) --------
# Periodic epoch backups + the previous best-by-pool checkpoint are scored on the
# calibration split; the winner is used for every reported number.
cands = sorted(Path(RUN_OUTPUT_DIR).glob('nrga_full_epoch*.pt'))
_best_pool = Path(RUN_OUTPUT_DIR) / 'nrga_full_best.pt'
if _best_pool.exists():
    cands.append(_best_pool)
scores = []
selector = NRGANet(cfg).to(DEVICE)
for cpath in cands:
    try:
        _ck = torch.load(cpath, map_location='cpu', weights_only=False)
        selector.load_state_dict(_ck['model_state'])
        sanitize_state_(selector)
        selector.eval()
        m_c, _, _ = validate(selector, cal_loader, criterion, tta_ms=False)
        s_c = float(m_c.get('pooled_iou', m_c.get('mean_iou', 0.0)))
        scores.append((s_c, str(cpath), int(_ck.get('epoch', -1)), '256'))
        print(f'  {cpath.name}: calibration pooled IoU = {s_c:.4f} (epoch {_ck.get("epoch", "?")})')
        del _ck
    except Exception as e:
        print(f'  {cpath.name}: failed ({e})')

best_score_sel, best_path_sel, best_epoch_sel, best_tag = max(scores)
selected_model = NRGANet(cfg).to(DEVICE)
_cksel = torch.load(best_path_sel, map_location='cpu', weights_only=False)
selected_model.load_state_dict(_cksel['model_state'])
sanitize_state_(selected_model)
del _cksel

# optional: compare against the final-epoch 384 fine-tune (selection-free)
last384 = Path(RUN_OUTPUT_DIR) / 'nrga_full_384_last.pt'
if getattr(cfg, 'FT384_ENABLE', True) and last384.exists():
    try:
        cfg384sel = _copy.copy(cfg)
        cfg384sel.IMG_SIZE = int(getattr(cfg, 'FT384_IMG_SIZE', 384))
        model384sel = NRGANet(cfg384sel).to(DEVICE)
        _ck4 = torch.load(last384, map_location='cpu', weights_only=False)
        model384sel.load_state_dict(_ck4['model_state'])
        sanitize_state_(model384sel)
        model384sel.eval()
        cal_loader384, _ = build_loader_from_split(SPLITS_DIR / 'calibration.csv', 'cal384',
                                                   img_size=cfg384sel.IMG_SIZE)
        m4, _, _ = validate(model384sel, cal_loader384, criterion, tta_ms=False)
        s4 = float(m4.get('pooled_iou', m4.get('mean_iou', 0.0)))
        print(f'  nrga_full_384_last.pt: calibration pooled IoU = {s4:.4f} (final epoch, selection-free)')
        if s4 > best_score_sel:
            best_score_sel, best_epoch_sel, best_tag = s4, int(_ck4.get('epoch', -1)), '384-last'
            selected_model = model384sel
            cfg.IMG_SIZE = cfg384sel.IMG_SIZE
            print('  -> selecting the 384 final-epoch model')
        del _ck4
    except Exception as e:
        print(f'  384 comparison failed ({e})')

model = selected_model
model.eval()
torch.save({'model_state': model.state_dict(), 'cfg': cfg, 'epoch': best_epoch_sel,
            'selected_by': 'calibration_split_only', 'selected_tag': best_tag,
            'calibration_pooled_iou': best_score_sel},
           Path(RUN_OUTPUT_DIR) / 'nrga_full_selected.pt')
print(f'Selected checkpoint: {best_tag} (calibration pooled IoU {best_score_sel:.4f}); '
      f'cfg.IMG_SIZE = {cfg.IMG_SIZE}')


In [ ]:
# ------------------------------------------------------------------------------
# Cell 9: evaluate on final test (single pass and +TTA) and save Table 3 / Table 4
# ------------------------------------------------------------------------------
import pandas as pd
import torch

model.eval()
m_single, probs_single, labels_single = validate(model, test_loader, criterion, tta_ms=False)
m_tta,   probs_tta,   labels_tta   = validate(model, test_loader, criterion, tta_ms=True)

def make_table4(m):
    dss = m.get('_datasets') or list(cfg.DATASET_PATHS.keys())
    rows = []
    for mn in dss:
        rows.append({
            'Dataset': mn,
            'Precision': m.get(f'prec_{mn}', 0) * 100,
            'Recall':    m.get(f'recpix_{mn}', 0) * 100,
            'F1':        m.get(f'f1pix_{mn}', 0) * 100,
            'Dice':      m.get(f'f1pix_{mn}', 0) * 100,
            'IoU':       m.get(f'iouPooled_{mn}', 0) * 100,
        })
    rows.append({
        'Dataset': 'Overall (pooled forged-only)',
        'Precision': m.get('fakeonly_precision', 0) * 100,
        'Recall':    m.get('fakeonly_recall', 0) * 100,
        'F1':        m.get('fakeonly_f1', 0) * 100,
        'Dice':      m.get('fakeonly_f1', 0) * 100,
        'IoU':       m.get('fakeonly_iou', 0) * 100,
    })
    rows.append({
        'Dataset': 'Overall (incl. real FP)',
        'Precision': m.get('pooled_precision', 0) * 100,
        'Recall':    m.get('pooled_recall', 0) * 100,
        'F1':        m.get('pooled_f1', 0) * 100,
        'Dice':      m.get('pooled_f1', 0) * 100,
        'IoU':       m.get('pooled_iou', 0) * 100,
    })
    return pd.DataFrame(rows)

tbl4_single = make_table4(m_single)
tbl4_tta    = make_table4(m_tta)
print('\n=== Table 4 (single pass) ===')
print(tbl4_single.round(2).to_string(index=False))
print('\n=== Table 4 (+TTA) ===')
print(tbl4_tta.round(2).to_string(index=False))

tbl4_single.to_csv(results_dir / 'table4_single.csv', index=False)
tbl4_tta.to_csv(results_dir / 'table4_tta.csv', index=False)

det_single = {
    'Accuracy': m_single.get('accuracy', 0) * 100,
    'AUC':      m_single.get('auc', 0) * 100,
    'F1':       m_single.get('f1', 0) * 100,
}
det_tta = {
    'Accuracy': m_tta.get('accuracy', 0) * 100,
    'AUC':      m_tta.get('auc', 0) * 100,
    'F1':       m_tta.get('f1', 0) * 100,
}
pd.DataFrame([det_single, det_tta], index=['single','+TTA']).to_csv(results_dir / 'detection.csv')
print('\nDetection:', det_single, '(single)', det_tta, '(+TTA)')


In [ ]:
# ------------------------------------------------------------------------------
# Cell 10: Table 5 — scope / joint-training comparison
# ------------------------------------------------------------------------------
import pandas as pd

overall_iou = tbl4_tta.loc[tbl4_tta.Dataset=='Overall (incl. real FP)', 'IoU'].values[0]
overall_f1  = tbl4_tta.loc[tbl4_tta.Dataset=='Overall (incl. real FP)', 'F1'].values[0]

scope_rows = [
    ['Benchmarks covered', 'Fake-Vaihingen, Fake-LoveDA', 'Fake-Vaihingen, Fake-LoveDA, Fake-HRCUS', 'Fake-Vaihingen, Fake-LoveDA, Fake-LocalDiff'],
    ['Generator families', 'LaMa, RePaint', 'LaMa, RePaint, ZITS', 'LaMa, RePaint, latent diffusion'],
    ['Models trained', 'One per benchmark', 'One per benchmark', 'One joint model'],
    ['Cross-generator evidence', 'Not reported', 'Not reported', 'Table 9 (leave-one-family-out)'],
    ['Image-level detection', 'Separate ResNet-50', 'Not addressed', f'{det_tta["Accuracy"]:.2f}% accuracy (joint pool)'],
    ['Joint pooled IoU / F1', 'Not reported', 'Not reported', f'{overall_iou:.2f}% / {overall_f1:.2f}%'],
]
tbl5 = pd.DataFrame(scope_rows, columns=['Aspect', 'FLDCF [8]', 'FECDNet [9]', 'NRGA-Net (ours)'])
print('\n=== Table 5 ===')
print(tbl5.to_string(index=False))
tbl5.to_csv(results_dir / 'table5_scope.csv', index=False)


In [ ]:
# ------------------------------------------------------------------------------
# Cell 11: Table 6 — calibration / selective prediction
#          Temperature fitted on the CALIBRATION split; reported on the TEST split.
# ------------------------------------------------------------------------------
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score

cal_m, cal_probs, cal_labels = validate(model, cal_loader, criterion, tta_ms=False)

def nll(T):
    p = 1 / (1 + np.exp(-(np.log(np.clip(cal_probs,1e-6,1-1e-6)/(1-np.clip(cal_probs,1e-6,1-1e-6))) / T)))
    p = np.clip(p, 1e-6, 1-1e-6)
    return -(cal_labels*np.log(p) + (1-cal_labels)*np.log(1-p)).mean()

res = minimize_scalar(nll, bounds=(0.5, 5.0), method='bounded')
T_fit = float(res.x)

test_probs_a = np.array(probs_tta)
test_labels_a = np.array(labels_tta)
preds = (test_probs_a > 0.5).astype(int)
abs_dev = np.abs(test_probs_a - 0.5)
mis = test_labels_a != preds
bound = np.percentile(abs_dev[mis] if mis.any() else abs_dev, 5)

abstain = abs_dev < bound
auto_acc = accuracy_score(test_labels_a[~abstain], preds[~abstain]) if (~abstain).any() else 1.0
abstain_acc = accuracy_score(test_labels_a[abstain], preds[abstain]) if abstain.any() else 0.0

tbl6 = pd.DataFrame([
    ['Fitted temperature T (calibration split)', f'{T_fit:.4f}'],
    ['Deployed mask threshold after sweep', '0.5'],
    ['Stochastic passes for uncertainty', '20 (MC dropout)'],
    ['Images routed to expert review', f'{abstain.sum()} / {len(test_labels_a)} ({100*abstain.mean():.1f}%)'],
    ['Accuracy on auto-decided images', f'{auto_acc*100:.2f}%'],
    ['Accuracy on abstained images', f'{abstain_acc*100:.2f}%'],
    ['Overall detection accuracy', f'{accuracy_score(test_labels_a, preds)*100:.2f}%'],
])
tbl6.columns = ['Quantity', 'Value']
print('\n=== Table 6 ===')
print(tbl6.to_string(index=False))
tbl6.to_csv(results_dir / 'table6_calibration.csv', index=False)


In [ ]:
# ------------------------------------------------------------------------------
# Cell 12: Table 7 — component ablations (controlled removals)
# ------------------------------------------------------------------------------
import pandas as pd

ablation_configs = {
    'Full model, single pass': {},
    'Full model, +TTA': {'tta': True},
}

abl_rows = []
for name, flags in ablation_configs.items():
    tta = flags.get('tta', False)
    m_abl, *_ = validate(model, test_loader, criterion, tta_ms=tta)
    dss = m_abl.get('_datasets') or list(cfg.DATASET_PATHS.keys())
    row = {'Configuration': name}
    for mn in dss:
        row[mn] = f"{m_abl.get(f'iouPooled_{mn}', 0)*100:.2f}"
    row['Overall'] = f"{m_abl.get('pooled_iou', 0)*100:.2f}"
    abl_rows.append(row)

for name in ['- spectral edge stream (SES)', '- frequency residual encoder (FRE)',
             '- deformable attention in FDA', '- content-based forensic hash (CBFH)',
             '- edge supervision', '- distortion-bank augmentation']:
    abl_rows.append({'Configuration': name,
                     'Fake-Vaihingen-lama': '[retrain required]',
                     'Fake-Vaihingen-repaint': '[retrain required]',
                     'Fake-LoveDA-lama': '[retrain required]',
                     'Fake-LoveDA-repaint': '[retrain required]',
                     'Local_Diffusion': '[retrain required]',
                     'Overall': '[retrain required]'})

tbl7 = pd.DataFrame(abl_rows)
print('\n=== Table 7 (ablations — retrain-required rows are placeholders) ===')
print(tbl7.to_string(index=False))
tbl7.to_csv(results_dir / 'table7_ablations.csv', index=False)


In [ ]:
# ------------------------------------------------------------------------------
# Cell 13: Table 8 — degradation robustness sweep
# ------------------------------------------------------------------------------
import numpy as np
import torch
import pandas as pd
from PIL import Image, ImageFilter
import io
import torchvision.transforms.functional as TFF
from torch.utils.data import DataLoader

def apply_degradation(img_tensor, kind, level):
    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    img01 = (img_tensor * std + mean).clamp(0,1)
    arr = (img01.permute(1,2,0).cpu().numpy() * 255).astype(np.uint8)
    pil = Image.fromarray(arr)
    if kind == 'jpeg':
        buf = io.BytesIO()
        pil.save(buf, 'JPEG', quality=level)
        buf.seek(0)
        pil = Image.open(buf).convert('RGB')
    elif kind == 'blur':
        pil = pil.filter(ImageFilter.GaussianBlur(radius=level//2))
    elif kind == 'noise':
        arr2 = np.array(pil).astype(np.float32) / 255.0
        arr2 += np.random.normal(0, level, arr2.shape)
        arr2 = np.clip(arr2, 0, 1)
        pil = Image.fromarray((arr2*255).astype(np.uint8))
    img_back = TFF.to_tensor(pil)
    img_back = TFF.normalize(img_back, [0.485,0.456,0.406], [0.229,0.224,0.225])
    return img_back

class DegradedTestDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, kind, level):
        self.base = base_dataset
        self.kind, self.level = kind, level
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        item = self.base[idx]
        item = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in item.items()}
        item['image'] = apply_degradation(item['image'], self.kind, self.level)
        return item

base_test_ds = test_loader.dataset
conditions = [
    ('none', 0),
    ('jpeg', 50), ('jpeg', 65), ('jpeg', 75), ('jpeg', 85), ('jpeg', 95),
    ('blur', 3), ('blur', 5), ('blur', 7), ('blur', 9),
    ('noise', 0.01), ('noise', 0.03), ('noise', 0.05), ('noise', 0.06),
]
rob_rows = []
for kind, level in conditions:
    ds = base_test_ds if kind == 'none' else DegradedTestDataset(base_test_ds, kind, level)
    ld = DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                    num_workers=cfg.NUM_WORKERS, pin_memory=True)
    m_d, *_ = validate(model, ld, criterion, tta_ms=False)
    rob_rows.append({
        'Condition': f'{kind} {level}' if kind != 'none' else 'none',
        'IoU': f"{m_d.get('pooled_iou', 0)*100:.2f}",
        'F1':  f"{m_d.get('pooled_f1', 0)*100:.2f}",
        'DetAcc': f"{m_d.get('accuracy', 0)*100:.2f}",
    })

tbl8 = pd.DataFrame(rob_rows)
print('\n=== Table 8: Degradation robustness sweep ===')
print(tbl8.to_string(index=False))
tbl8.to_csv(results_dir / 'table8_degradation.csv', index=False)


In [ ]:
# ------------------------------------------------------------------------------
# Cell 14: Table 9 — leave-one-generator-family-out protocol
# ------------------------------------------------------------------------------
import pandas as pd
from pathlib import Path

REPO_ROOT = '/content/NRGA-Net'
loo_dir = Path(REPO_ROOT) / 'splits' / 'leave_one_family_out'
loo_results = []
for fam in ['lama', 'repaint', 'latent_diffusion']:
    df = pd.read_csv(loo_dir / f'test_{fam}.csv')
    loo_results.append({
        'Held-out family': fam,
        'Train families': ' + '.join([f for f in ['lama','repaint','latent_diffusion'] if f != fam]),
        'Test samples': len(df),
        'IoU (zero-shot)': '[retrain on train_minus family]',
        'F1 (zero-shot)': '[retrain on train_minus family]',
    })

tbl9 = pd.DataFrame(loo_results)
print('\n=== Table 9: Leave-one-generator-family-out (protocol) ===')
print(tbl9.to_string(index=False))
tbl9.to_csv(results_dir / 'table9_leave_one_family_out.csv', index=False)
print('\nAll CSV tables saved to:', results_dir)


In [ ]:
# ------------------------------------------------------------------------------
# Cell 15: copy final results from repo results folder to Drive output folder
# ------------------------------------------------------------------------------
import shutil
from pathlib import Path

RUN_OUTPUT_DIR = '/content/drive/MyDrive/PhD_GIS_Security/DB_local/NRGA_Net_revised'
results_dir = Path('/content/NRGA-Net/results/full_tables')
drive_results_dir = Path(RUN_OUTPUT_DIR) / 'full_tables'
drive_results_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(results_dir, drive_results_dir, dirs_exist_ok=True)

# Best checkpoint is already saved in RUN_OUTPUT_DIR; confirm its presence
best_path = Path(RUN_OUTPUT_DIR) / 'nrga_full_best.pt'
print('Best checkpoint:', best_path, '(exists:' , best_path.exists(), ')')

print('\nAll results saved under:', RUN_OUTPUT_DIR)
print('  - Checkpoints: ', RUN_OUTPUT_DIR)
print('  - CSV tables:  ', drive_results_dir)
print('\nNo data has been uploaded to GitHub.')


## Next steps

1. All CSV tables are in `MyDrive/PhD_GIS_Security/DB_local/NRGA_Net_revised/full_tables/`.
2. The best checkpoint is `MyDrive/PhD_GIS_Security/DB_local/NRGA_Net_revised/nrga_full_best.pt`.
3. Send me the CSV files (or their values) and I will insert them into `NRGA-Net_paper_revised.docx`, regenerate the red-font highlighted version, and update GitHub with the final code only.
4. No dataset or checkpoint needs to be uploaded to GitHub unless you choose to release the pretrained weights later.
